In [1]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanumsearch-embs")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/384 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1043 [00:00<?, ? examples/s]

In [2]:
import numpy as np

def parse_fast(example):
    arr = np.fromstring(example["embedding"].strip("[]"), sep=", ", dtype=np.float32)
    example["embedding"] = arr.tolist()
    return example

dataset = dataset.map(parse_fast)

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

In [3]:
# from sentence_transformers import (
#     SentenceTransformer,
# )

# model = SentenceTransformer('pameydorke/arcanum-cross-platform-retriever')

In [4]:
import torch

embeddings = torch.tensor(dataset["train"]["embedding"], dtype=torch.float32)

In [5]:
import torch.nn.functional as F

def retrieve(query, top_k=5):
    query_embs = model.encode(query, convert_to_tensor=True)

    similarities = F.cosine_similarity(query_embs, embeddings)

    scores, indices = torch.topk(similarities, k=top_k)

    return [
        {"score": float(scores[i]), "url": dataset["train"][indices[i]]["url"], "text": dataset["train"][indices[i]]["content"]}
        for i in range(top_k)
    ]

In [6]:
# TODO: recall, ndcg